In [4]:
import pandas as pd
import numpy as np
import os
import pickle
import json
# for content_based filtering
from sklearn.metrics.pairwise import cosine_similarity

# for collabaractive filtering

In [5]:
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')
tmdb_movies_info = pd.read_csv('../dataset/processed/movies_df_clean.csv')



In [6]:
with open('../dataset/processed/userWatched_movies.json','r') as file:
    userWatched_movies = json.load(file)
userWatched_movies = pd.Series(userWatched_movies)

In [7]:
#  filter movies which have rated by users. 
tmdb_movies_info = tmdb_movies_info[tmdb_movies_info['movie_id'].isin(movieId_lookup['tmdb_movieId'])].reset_index(drop=True)

In [8]:
# save tmdb_movies_to database.
tmdb_movies_info.to_csv('../dataset/processed/tmdb_movies_info.csv',index=False)

In [9]:
with open('../artifact/svd_model.pkl','rb') as file:
    svd_model = pickle.load(file)

In [10]:
combined_embeddings = np.load('../artifact/embeddings/final_embeddings.npy')
director_embeddings = np.load('../artifact/embeddings/director_embeddings.npy')
keyword_embeddings = np.load('../artifact/embeddings/keyword_embeddings.npy')
genre_embeddings = np.load('../artifact/embeddings/genre_embeddings.npy')
overview_embeddings = np.load('../artifact/embeddings/overview_embeddings.npy')


In [11]:

def content_based_filtering(movie_idx):
    '''
    apply content based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    '''
   
    # find the index of tmdb movie based on index value
    tmdb_movieId = movieId_lookup.iloc[movie_idx]['tmdb_movieId']
    
    # calculate movie similarity based on embedded text vector metrix.
    tmdb_index = tmdb_movies_info[tmdb_movies_info['movie_id'] == tmdb_movieId].index[0]
    similarities = cosine_similarity([combined_embeddings[tmdb_index]],combined_embeddings)[0]

    # sort index based on values then change the order into descending and give first 10 values index
    top_indicies = similarities.argsort()[::-1][:11]

    # # calculate score of movide matching
    top_similar_movies  = []
    for  rm in top_indicies[1:]:
        
        movie_info = {'movie_title' : tmdb_movies_info.loc[rm,'title'],'index' :rm}
        
        movie_similarity = get_similarity_explaination(tmdb_index,rm)
        
        movie_info.update(movie_similarity)
        
        top_similar_movies.append(movie_info)

        

    return top_similar_movies
    
    # for similary_tmdb_id in top_indicies:
    # for movie_title in  tmdb_movies_info.loc[top_indicies,'title']:
    #     print(movie_title)
    # return tmdb_movies_info.loc[top_indicies,'title'],match_score
    
    
    return top_indicies
    

In [12]:


def get_similarity_explaination(movie_idx,remm_idx):
    gen_sim = cosine_similarity(genre_embeddings[movie_idx].reshape(1,-1),genre_embeddings[remm_idx].reshape(1,-1))[0,0]

    overview_sim = cosine_similarity(overview_embeddings[movie_idx].reshape(1,-1),overview_embeddings[remm_idx].reshape(1,-1))[0,0]

    keyword_sim = cosine_similarity(keyword_embeddings[movie_idx].reshape(1,-1),keyword_embeddings[remm_idx].reshape(1,-1))[0,0]

    final_score =(gen_sim*30 + overview_sim*40 + keyword_sim  *30 )
    

    return {
            'genre_similarity' : np.round(gen_sim, 2),
            "keyword_similarity": np.round(keyword_sim,2),
            "overview_similarity": np.round((overview_sim*100),2),      
            "final_score": np.round(final_score,2)
            }
    

In [13]:
content_based_filtering(current_movie_idx)[3]['overview_similarity']

NameError: name 'current_movie_idx' is not defined

In [14]:
current_movie_idx = 76
match_score = content_based_filtering(current_movie_idx) # movie index of current movie

recommended_movies_info = []
for recommended_movie in match_score:
    rm = ContentRecommendationMovie(recommended_movie['index'],recommended_movie['overview_similarity'])
    rm.calcualte_similarities(current_movie_idx)

    recommended_movies_info.append(rm)

for rmc in recommended_movies_info:
    print(f'Movie Title is : {rmc.movie_title}')
    print(f'Movie matching_keywords is : {rmc.matching_keywords}')
    print(f'Movie matching_gernes is : {rmc.matching_gernes}')
    print(f'Movie overview_similarity is : {rmc.overview_similarity}')
    print(f'Movie matching_director is : {rmc.matching_director}')
    print('-'*25)


NameError: name 'ContentRecommendationMovie' is not defined

In [15]:
class ContentRecommendationMovie:
    '''
     Save information and find similaraties and explaination of recommendated movie
    '''
    def __init__(self,movie_idx,overview_similarity):
        '''
            Input : Index of recommendated movie object ( tmdb table index )
        '''
        self.movie_idx = movie_idx
        
        self.movie_title = None
        self.matching_keywords = None
        self.matching_gernes = None
        self.matching_director = None
        self.overview_similarity = round(overview_similarity,2)
   
    def calcualte_similarities(self,recommended_tmdb_idx):
        '''
            Calcluate the similarities between current movies information and given movies.
            and store it data into data members.
        '''

        cm = tmdb_movies_info.loc[self.movie_idx]
        rm = tmdb_movies_info.loc[recommended_tmdb_idx]
       
        
        # keywords 
        cm_keys = set(cm['keywords_clean'].strip().split(', '))
        rm_keys = set(rm['keywords_clean'].strip().split(', '))
        
        matching_keywords = list(rm_keys & cm_keys )
        matching_keywords.sort()
        
        # genres
        cm_genres = set(cm['genres_clean'].strip().split(', '))
        rm_genres = set(rm['genres_clean'].strip().split(', '))
    
        matching_gernes = list(cm_genres & rm_genres)
        matching_gernes.sort()
    
        # director matching
        cm_director = cm['crew_clean'].strip().split('Director: ')[1]
        rm_director = rm['crew_clean'].strip().split('Director: ')[1]
    
        if cm_director == rm_director:
            matching_director = True
        else:
            matching_director = False
    
        
        # Movie Overview

        
        self.movie_title = cm['title']
        self.matching_keywords = matching_keywords
        self.matching_gernes   = matching_gernes
        self.matching_director = matching_director

        return {
             'title'             : self.movie_title,
             'matching_keywords' :  self.matching_keywords,
             'matching_gernes'   : self.matching_gernes,
             'matching_director' : self.matching_director,
             'overview_similarity' : self.overview_similarity
        }
    
        

In [16]:
def calcualte_similarities(current_idx,recommended_tmdb_idx):
    
    cm = tmdb_movies_info.loc[current_idx]
    rm = tmdb_movies_info.loc[recommended_tmdb_idx]

    # keywords 
    cm_keys = set(cm['keywords_clean'].strip().split(', '))
    rm_keys = set(rm['keywords_clean'].strip().split(', '))
    
    matching_keywords = list(rm_keys & cm_keys )
    matching_keywords.sort()

    # genres
    cm_genres = set(cm['genres_clean'].strip().split(', '))
    rm_genres = set(rm['genres_clean'].strip().split(', '))

    matching_gernes = list(cm_genres & rm_genres)
    matching_gernes.sort()

    # director matching
    cm_director = cm['crew_clean'].strip().split('Director: ')[1]
    rm_director = rm['crew_clean'].strip().split('Director: ')[1]

    if cm_director == rm_director:
        matching_director = True
    else:
        matching_director = False

    # Movie Overview

    return  {
             
             'matching_keywords' :  matching_keywords,
             'matching_gernes'   : matching_gernes,
             'matching_director' : matching_director
        }
    

In [17]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

movielens_movies = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))


In [25]:
def collaborative_based_filtering(user_id):
    '''
    apply collaborative based filtering on given movie index and return the top similarities movies.
    input:
    user_id : moviesLens rating user_id which we want to recommand movies.

    return :
    top similarity movies list.
    ''' 

     # find the index of movielens movie based on index value
    # get all movies which exist in dataset.
    all_movies = set(movieId_lookup['movieLens_movieId'].tolist())
    
    movies_watched = set(userWatched_movies[str(user_id)])
    
    
    if not type(movies_watched) == set:
        movie_watched = set(movie_watched)
        
    unwatched_movies = all_movies - movies_watched

    
    predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
    for movie in unwatched_movies:
        pred_rating = svd_model.predict(user_id,movie).est
        predictions.append([movie,pred_rating])
       
    
    # sort the predicted rating by highly rated movie prediction
    predictions.sort(key=lambda x : x[1],reverse=True)


    
    movie_lookup = dict(zip(movieId_lookup['movieLens_movieId'],movieId_lookup['title_clean']))
    
    for movieId, rating in predictions[:11]:
        print(movie_lookup[movieId],' : ',rating)
    

    
    

In [26]:
collaborative_based_filtering(519)

Harry potter and the half-blood prince  :  4.522764175597403
Fear and loathing in las vegas  :  4.2696316058748645
Gran torino  :  4.262284979252038
Hotel rwanda  :  4.242269200661832
Lost in translation  :  4.219929579008488
Blood diamond  :  4.138431951574938
Welcome to the dollhouse  :  4.129870837309026
Cinderella man  :  4.122785159070314
American history x  :  4.094437673941751
Mission: impossible - ghost protocol  :  4.083773602015143
Children of men  :  4.0837487649619435


In [ ]:
content_based_filtering(movieId_lookup.sample(1).index[0])

In [131]:
movie_id = movieId_lookup.sample(1).index[0]
user_id = '108795'
print(f' movie_id {movie_id}')
print(f' user_id {user_id}')

 movie_id 1279
 user_id 108795


In [137]:
content_based_filtering(movie_id)

1279                                         Ironclad
186                                 Kingdom of Heaven
2043                                          Henry V
354                                      First Knight
524     In the Name of the King: A Dungeon Siege Tale
12        Pirates of the Caribbean: On Stranger Tides
575                         Elizabeth: The Golden Age
2420                                    Hell's Angels
780                          The Last of the Mohicans
1890                                     Barry Lyndon
1414                                        Ladyhawke
Name: title, dtype: str

In [133]:
collaborative_based_filtering(user_id)

One flew over the cuckoo's nest  :  4.253267246770918
Apocalypse now  :  4.235958151441624
It happened one night  :  4.226346622163668
Dr. strangelove or: how i learned to stop worrying and love the bomb  :  4.197599273246513
Some like it hot  :  4.172173785547581
Rebecca  :  4.166519797483818
Midnight cowboy  :  4.164186285028516
Henry v  :  4.162900041991528
Monty python and the holy grail  :  4.156676832042193
Taxi to the dark side  :  4.154126559409125
Lawrence of arabia  :  4.143300164788154


In [130]:
userWatched_movies.index

Index(['519', '4019', '5807', '6862', '13803', '24236', '29179', '30178',
       '37824', '40491', '43091', '44772', '46546', '47156', '48541', '49727',
       '51278', '52020', '57743', '57826', '60461', '61941', '66926', '69502',
       '71185', '75154', '79045', '87418', '88392', '91338', '96408', '102957',
       '107409', '108795', '109459', '111724', '111770', '113946', '114613',
       '119499', '120525', '129396', '130379', '144475', '147756', '148441',
       '149753', '156131', '158866', '161939'],
      dtype='str')